# Export sensory neuron skeletons as SWC

Skeletonizes each sensory axon with `pcg_skel.pcg_skeleton` (level-2 skeletons over the
PyChunkedGraph) and writes one SWC per neuron to `../../data/sensory_neuron_swc/`, named
`<tag>_<root_id>.swc`.

Those SWC files ship with the repository, so this only needs re-running to rebuild them —
for instance against a newer materialization. **Requires CAVE access.**

The exports are in 32×32×45 nm voxel units; `figure3jk_skeleton_synapse_maps.ipynb`
converts between these and the 16×16×45 nm units of the synapse table.


In [ ]:
# Libraries
import caveclient
import subprocess
import json
import numpy as np
import matplotlib.pyplot as plt
# Setup caveclient
client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr", ) #server_address='https://global.daf-apis.com')

In [ ]:
sensory_tag = client.materialize.query_table('sensory_neuron_label')

In [ ]:
sensory_tag

In [ ]:
from pathlib import Path
from typing import Sequence, Optional, Tuple
import numpy as np
import caveclient
import pcg_skel

# ---- Config (adjust as needed) ----
DATASTACK = "wclee_mouse_spinalcord_cltmr"
RES_NM_PER_PX = (32.0, 32.0, 45.0)        # (x,y,z) nm/px
SOMA_COLLAPSE_RADIUS_NM = 7500
DEFAULT_RADIUS_NM = 1000.0                 # in nm
SWC_UNITS = "nm"                           # "nm" | "um" | "px"

# ---- Helpers ----------------------------------------------------------
def _px_to_nm(v_px: np.ndarray, res_nm_per_px: Sequence[float]) -> np.ndarray:
    return np.asarray(v_px, float) * np.asarray(res_nm_per_px, float)[None, :]

def _nm_to_um(v_nm: np.ndarray) -> np.ndarray:
    return np.asarray(v_nm, float) / 1000.0

def _choose_root_idx(verts_px: np.ndarray,
                     soma_nm: Optional[Sequence[float]],
                     res_nm_per_px: Sequence[float]) -> int:
    if soma_nm is None:
        return 0
    verts_nm = _px_to_nm(verts_px, res_nm_per_px)
    soma_nm = np.asarray(soma_nm, float).reshape(1, 3)
    d2 = np.sum((verts_nm - soma_nm) ** 2, axis=1)
    return int(np.argmin(d2))

def _bfs_parent(n_vertices: int, edges: np.ndarray, root_idx: int) -> np.ndarray:
    edges = np.asarray(edges, dtype=int)
    adj = [[] for _ in range(n_vertices)]
    for i, j in edges:
        adj[i].append(j); adj[j].append(i)
    parent = np.full(n_vertices, -2, dtype=int)
    parent[root_idx] = -1
    q = [root_idx]
    while q:
        u = q.pop(0)
        for v in adj[u]:
            if parent[v] == -2:
                parent[v] = u
                q.append(v)
    parent[parent == -2] = -1
    return parent

def _coords_in_units(verts_px: np.ndarray,
                     units: str,
                     res_nm_per_px: Sequence[float],
                     default_radius_nm: float) -> Tuple[np.ndarray, np.ndarray]:
    if units == "nm":
        coords = _px_to_nm(verts_px, res_nm_per_px)
        radii = np.full(len(verts_px), default_radius_nm, float)
    elif units == "um":
        coords = _nm_to_um(_px_to_nm(verts_px, res_nm_per_px))
        radii = np.full(len(verts_px), default_radius_nm / 1000.0, float)
    elif units == "px":
        coords = np.asarray(verts_px, float)
        avg_res = float(np.mean(np.asarray(res_nm_per_px, float)))
        radii = np.full(len(verts_px), default_radius_nm / avg_res, float)
    else:
        raise ValueError("units must be one of {'nm','um','px'}")
    return coords, radii

def _fetch_skeleton(client, root_id: int,
                    soma_nm: Optional[Sequence[float]],
                    res_nm_per_px: Sequence[float],
                    collapse_radius_nm: int):
    """
    Fetch a skeleton. If soma is provided, root & collapse near soma.
    Returns vertices in voxels (px) and undirected edges (Nx2).
    """
    if soma_nm is not None:
        skel, mesh, _ = pcg_skel.pcg_skeleton(
            root_id,
            client,
            return_mesh=True,
            root_point=np.asarray(soma_nm, float),
            root_point_resolution=tuple(res_nm_per_px),
            collapse_soma=True,
            collapse_radius=int(collapse_radius_nm),
            return_l2dict=True,
        )
    else:
        skel = pcg_skel.pcg_skeleton(root_id, client, return_mesh=False)
    return np.asarray(skel.vertices, float), np.asarray(skel.edges, int)

# ---- One-shot export of CLEANED (drop max-X) axon-only SWC -------------------
def export_cleaned_neuron_as_axon_swc(
    root_id: int,
    out_path: str,
    *,
    soma_px: Optional[Sequence[float]] = None,   # optional soma (x,y,z) in px
    soma_nm: Optional[Sequence[float]] = None,   # optional soma (x,y,z) in nm
    datastack: str = DATASTACK,
    res_nm_per_px: Sequence[float] = RES_NM_PER_PX,
    units: str = SWC_UNITS,
    default_radius_nm: float = DEFAULT_RADIUS_NM,
    collapse_radius_nm: int = SOMA_COLLAPSE_RADIUS_NM,
    print_info: bool = True,
) -> str:
    """
    Export a single neuron skeleton as a CLEANED SWC:
      1) Fetch skeleton (optionally collapse near soma).
      2) Root via BFS from soma-nearest vertex (or index 0 if no soma).
      3) Convert coords to units; set all node types = 2 (axon).
      4) CLEAN: remove the node with the largest X (in output units),
         detach its direct children (parent=-1), drop the node, and renumber 1..N.
      5) Write the final SWC to 'out_path'.

    Returns the output path.
    """
    client = caveclient.CAVEclient(datastack)

    # normalize soma to nm (if provided)
    if soma_nm is not None:
        soma_nm_arr = np.asarray(soma_nm, float)
    elif soma_px is not None:
        soma_nm_arr = np.asarray(soma_px, float) * np.asarray(res_nm_per_px, float)
    else:
        soma_nm_arr = None

    # fetch skeleton and pick root
    verts_px, edges = _fetch_skeleton(client, root_id, soma_nm_arr, res_nm_per_px, collapse_radius_nm)
    if len(verts_px) == 0:
        raise ValueError("Fetched skeleton has no vertices.")
    root_idx = _choose_root_idx(verts_px, soma_nm_arr, res_nm_per_px)
    parent = _bfs_parent(len(verts_px), edges, root_idx)

    # convert coordinates/radii to output units
    coords, radii = _coords_in_units(verts_px, units, res_nm_per_px, default_radius_nm)

    # ---------- CLEAN: drop max-X node, detach its children, renumber ----------
    i_max = int(np.argmax(coords[:, 0]))  # index of max-X node
    removed_n_id = i_max + 1  # n-id before cleaning (for note)
    removed_row = {
        "n": removed_n_id,
        "x": float(coords[i_max, 0]),
        "y": float(coords[i_max, 1]),
        "z": float(coords[i_max, 2]),
        "radius": float(radii[i_max]),
        "parent": int(parent[i_max]),
        "type": 2,
    }

    # Detach direct children of the removed node
    child_mask = (parent == i_max)
    num_children_detached = int(np.sum(child_mask))
    parent = parent.copy()
    parent[child_mask] = -1

    # Build mask to drop the node
    keep_mask = np.ones(len(coords), dtype=bool)
    keep_mask[i_max] = False

    # Adjust parent indices after removal:
    #   -1 stays -1; indices > i_max shift down by 1; i_max itself is gone.
    parent_keep = parent[keep_mask]
    parent_keep = np.where(parent_keep == -1, -1, parent_keep - (parent_keep > i_max))

    # Apply mask to coords/radii
    coords_keep = coords[keep_mask]
    radii_keep = radii[keep_mask]

    # Renumber to 1..N in current order
    n_new = np.arange(1, len(coords_keep) + 1, dtype=int)

    # ---------- Write final SWC ----------
    out_path = str(out_path)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        f.write("# Single-neuron SWC (ALL nodes type=2 / axon) — CLEANED (drop max-X)\n")
        f.write(f"# root_id={root_id}\n")
        f.write(f"# Datastack: {datastack}\n")
        f.write(f"# Units: {units}\n")
        f.write(f"# Resolution (nm/px): {tuple(float(x) for x in res_nm_per_px)}\n")
        f.write("# Columns: n type x y z radius parent\n")
        f.write("# Clean note: removed node with largest X; detached its children (parent=-1); renumbered 1..N\n")
        f.write(f"# Removed: old_n={removed_row['n']}, x={removed_row['x']:.3f}, y={removed_row['y']:.3f}, "
                f"z={removed_row['z']:.3f}, radius={removed_row['radius']:.3f}, parent={removed_row['parent']}\n")
        f.write(f"# Children detached: {num_children_detached}\n")

        for i in range(len(coords_keep)):
            x, y, z = coords_keep[i].tolist()
            r = float(radii_keep[i])
            p = int(parent_keep[i])
            p_out = -1 if p == -1 else (p + 1)  # convert 0-based index to SWC id
            f.write(f"{int(n_new[i])} 2 {x:.3f} {y:.3f} {z:.3f} {r:.3f} {p_out}\n")

    if print_info:
        print(f"[SWC] wrote CLEANED skeleton ({len(coords_keep)} nodes) → {out_path}")
        print("  Removed node (pre-clean numbering):", removed_row)
        print(f"  Children detached: {num_children_detached}")
        # Sanity: show new max-X row
        i_new_max = int(np.argmax(coords_keep[:, 0]))
        print("  New max-X row:", {
            "n": int(n_new[i_new_max]), "type": 2,
            "x": float(coords_keep[i_new_max, 0]),
            "y": float(coords_keep[i_new_max, 1]),
            "z": float(coords_keep[i_new_max, 2]),
            "radius": float(radii_keep[i_new_max]),
            "parent": int(-1 if parent_keep[i_new_max] == -1 else parent_keep[i_new_max] + 1),
        })

    return out_path


In [ ]:
from pathlib import Path
from typing import List, Optional, Sequence
import os, re
import numpy as np
import pandas as pd

# --- helpers ---
def _slug_tag(s):
    s = str(s).strip().lower()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^a-z0-9_\-]", "", s)
    return s or "untagged"

def _parse_soma_px(val) -> Optional[Sequence[float]]:
    if val is None:
        return None
    if isinstance(val, (list, tuple, np.ndarray)) and len(val) == 3:
        return [float(val[0]), float(val[1]), float(val[2])]
    if isinstance(val, str):
        s = val.strip().strip("[]()")
        parts = re.split(r"[,\s]+", s)
        if len(parts) >= 3:
            return [float(parts[0]), float(parts[1]), float(parts[2])]
    # allow NaN-like
    try:
        if np.all(pd.isna(val)):
            return None
    except Exception:
        pass
    raise ValueError(f"Cannot parse pt_position → soma_px from: {val!r}")

def batch_export_cleaned_axons(
    root_ids: List[int],
    sensory_tag: pd.DataFrame,
    out_dir: str,
    *,
    datastack: str = "wclee_mouse_spinalcord_cltmr",
    res_nm_per_px: Sequence[float] = (32.0, 32.0, 45.0),
    units: str = "nm",
    default_radius_nm: float = 1000.0,
    collapse_radius_nm: int = 7500,
    print_info_each: bool = False,
) -> pd.DataFrame:
    """
    For each root_id:
      - look up tag and pt_position (as soma_px) in sensory_tag
      - export cleaned axon-only SWC named: {tag}_{root_id}.swc
    Returns a log DataFrame.
    Requires: export_cleaned_neuron_as_axon_swc(...) in scope.
    """
    out_dir = str(out_dir)
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    log = []
    idx = sensory_tag.set_index("pt_root_id", drop=False)

    for rid in root_ids:
        row = idx.loc[rid] if rid in idx.index else None
        if row is None or (isinstance(row, pd.DataFrame) and row.empty):
            log.append(dict(root_id=rid, tag=None, out_path=None, status="no_match", error="pt_root_id not in sensory_tag"))
            continue
        r = row.iloc[0] if isinstance(row, pd.DataFrame) else row
        tag = r.get("tag", "untagged")
        soma_raw = r.get("pt_position", None)

        try:
            soma_px = _parse_soma_px(soma_raw) if soma_raw is not None else None
        except Exception as e:
            log.append(dict(root_id=rid, tag=tag, out_path=None, status="bad_pt_position", error=str(e)))
            continue

        fname = f"{_slug_tag(tag)}_{rid}.swc"
        out_path = os.path.join(out_dir, fname)

        try:
            export_cleaned_neuron_as_axon_swc(
                rid,
                out_path,
                soma_px=soma_px,
                datastack=datastack,
                res_nm_per_px=res_nm_per_px,
                units=units,
                default_radius_nm=default_radius_nm,
                collapse_radius_nm=collapse_radius_nm,
                print_info=print_info_each,
            )
            log.append(dict(root_id=rid, tag=tag, out_path=out_path, status="ok", error=None))
        except Exception as e:
            log.append(dict(root_id=rid, tag=tag, out_path=out_path, status="error", error=str(e)))

    return pd.DataFrame(log)


In [ ]:
root_ids = [
    720575940891163574, 720575940922958792,
    720575940901960430, 720575940920833705,
    720575940893392149, 720575940902363320,
    720575940881926778, 720575940932756628,
    720575940883866908, 720575940882155492,
    720575940847552365, 720575940897347842,
    720575940918288693, 720575940881918801,
    720575940883734520, 720575940927175960,
]

out_dir = "../data/sensory_neuron_swc"

log_df = batch_export_cleaned_axons(
    root_ids,
    sensory_tag,
    out_dir,
    datastack="wclee_mouse_spinalcord_cltmr",
    res_nm_per_px=(32.0, 32.0, 45.0),
    units="nm",
    default_radius_nm=1000.0,
    collapse_radius_nm=7500,
    print_info_each=False,   # change to True for per-neuron verbose info
)

print(log_df)
# Optionally save:
# log_df.to_csv(os.path.join(out_dir, "batch_export_log.csv"), index=False)
